# Proyecto #1: Biodiversity at Scale
## Parte 5: Regularización, Generalización y Estudio de Ablación
**Maestría de Investigación en IA - UTEC Posgrado**  
**Curso**: Aprendizaje Profundo – Práctica  
**Profesor**: Dra. Aurea Soriano-Vargas

---
### Objetivos de esta etapa:
1. Evaluar experimentalmente técnicas contra el sobreajuste: **Batch Normalization, Dropout, Data Augmentation, Weight Decay y Early Stopping**.
2. Ejecutar un **Ablation Study** sistemático conforme al diseño de la Sección 9.4 de la rúbrica:

| Exp | BatchNorm | Dropout | Augmentation | Weight Decay |
| :---: | :---: | :---: | :---: | :---: |
| **E1** | No | No | No | No |
| **E2** | Sí | No | No | No |
| **E3** | Sí | Sí | No | No |
| **E4** | Sí | Sí | Sí | No |
| **E5** | Sí | Sí | Sí | Sí |

3. Identificar rigurosamente qué componentes aportan valor real a la generalización.


In [ ]:
import sys
from pathlib import Path
ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))

import torch
from src.utils.seed import seed_everything
from src.data.dataset import SyntheticINatDataset
from src.data.dataloader import build_dataloaders
from src.models.cnn_custom import MiniINatCNN
from src.training.losses import build_criterion
from src.training.optimizers import build_optimizer
from src.training.early_stopping import EarlyStopping
from src.training.trainer import Trainer
from src.utils.tracking import ExperimentTracker

SEED = 42
seed_everything(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tracker = ExperimentTracker(log_dir=str(ROOT_DIR / "logs"))


### 1. Ejecución del Estudio de Ablación Controlado


In [ ]:
ablation_configs = [
    {"name": "E6_Ablation_1_None", "bn": False, "drop": 0.0, "aug": "none", "wd": 0.0, "desc": "Sin Regularización"},
    {"name": "E6_Ablation_2_BN", "bn": True, "drop": 0.0, "aug": "none", "wd": 0.0, "desc": "+ BatchNorm"},
    {"name": "E6_Ablation_3_BN_Drop", "bn": True, "drop": 0.3, "aug": "none", "wd": 0.0, "desc": "+ Dropout"},
    {"name": "E6_Ablation_4_BN_Drop_Aug", "bn": True, "drop": 0.3, "aug": "standard", "wd": 0.0, "desc": "+ Data Augmentation"},
    {"name": "E6_Ablation_5_All", "bn": True, "drop": 0.3, "aug": "standard", "wd": 1e-4, "desc": "+ Weight Decay"}
]

N_CLASSES = 50
BATCH_SIZE = 32
EPOCHS = 4

train_ds = SyntheticINatDataset(num_samples=50 * 35, num_classes=N_CLASSES, img_size=64, seed=SEED)
val_ds = SyntheticINatDataset(num_samples=50 * 10, num_classes=N_CLASSES, img_size=64, seed=SEED+1)
train_loader, val_loader = build_dataloaders(train_ds, val_ds, batch_size=BATCH_SIZE, num_workers=2, seed=SEED)
criterion = build_criterion("cross_entropy")

for cfg in ablation_configs:
    print(f"\n--- Ejecutando {cfg['name']} ({cfg['desc']}) ---")
    model = MiniINatCNN(num_classes=N_CLASSES, use_batchnorm=cfg["bn"], dropout_rate=cfg["drop"])
    opt = build_optimizer(model, opt_type="adamw", lr=1e-3, weight_decay=cfg["wd"])
    
    es = EarlyStopping(patience=3, metric_name="val_macro_f1")
    trainer = Trainer(model, criterion, opt, device=device, early_stopping=es, use_amp=True)
    
    hist, best_m, vram, t_time = trainer.fit(train_loader, val_loader, epochs=EPOCHS, verbose=False)
    
    reg_str = f"{'BN ' if cfg['bn'] else ''}{'Drop ' if cfg['drop']>0 else ''}{'WD' if cfg['wd']>0 else ''}".strip() or "None"
    tracker.log_experiment(
        exp_id=cfg["name"],
        model_name="MiniINatCNN",
        optimizer="AdamW",
        regularization=reg_str,
        augmentation=cfg["aug"],
        transfer_learning="No",
        long_tail="No",
        metrics=best_m,
        training_time_sec=t_time,
        peak_vram_mb=vram,
        param_count_m=0.15
    )
    print(f"Resultado: Val Top-1 = {best_m['top1_acc']*100:.2f}% | Val Macro F1 = {best_m['macro_f1']*100:.2f}%")


### 2. Tabla Comparativa del Estudio de Ablación


In [ ]:
df = tracker.to_dataframe()
ablation_df = df[df["exp_id"].str.contains("Ablation")]
print(ablation_df[["exp_id", "regularization", "augmentation", "top1_acc", "macro_f1", "time_sec"]].to_markdown(index=False))
